In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.transform import Rotation as R, RotationSpline,Slerp
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import os
from onb_analysis import *
from sklearn.cluster import KMeans

In [ ]:
df_all = pd.read_csv("20250905_all_df_summary.csv")
df_all["DateTime"] = pd.to_datetime(df_all["DateTime"])
df_all["Date"] = df_all["DateTime"].dt.date
df_all = df_all.reset_index()

In [ ]:
df_all["lamina_label"] = [str(dt)+"_"+str(ln) for dt,ln in zip(df_all["Date"],df_all["lamina no."])]
df_all.iloc[:5,-3:]

In [ ]:
df_all2 = df_all.copy()
df_all2["PD Elev (deg)"] = df_all2["basal_vec_z2"].apply(np.arcsin).apply(np.degrees)
df_all2["AdAb Elev (deg)"] = df_all2["bv_w_z2"].apply(np.arcsin).apply(np.degrees)
df_all2["ML Elev (deg)"] = df_all2["pca2_z2"].apply(np.arcsin).apply(np.degrees)
df_all2["PD Azim (deg)"] = np.degrees(np.arctan2(df_all2["basal_vec_y2"],df_all2["basal_vec_x2"]))

In [ ]:
label_list = ["PD Elev (deg)","AdAb Elev (deg)","ML Elev (deg)","PD Azim (deg)"]

plt.rcParams["font.size"] = 18
fig,axes = plt.subplots(figsize=(21,4),ncols=4,nrows=1)
plt.subplots_adjust(wspace=0.4)
axes = axes.ravel()
for i,l in enumerate(label_list):
    ax = axes[i]
    sns.lineplot(
        data=df_all2,
        x="Duration(min)",
        y=l,
        hue="lamina_label",
        style="lamina_label",
        markers=True,
        legend=None,
        ax=ax
    )
#fig.savefig("graph/20250922_grav_reori_elev_angles_lineplot.svg",format="svg",
#            bbox_inches="tight")
plt.show()

In [ ]:
succeed_idx = df_all2[df_all2["AdAb Elev (deg)"]>0]["lamina_label"].unique()

In [ ]:
df_list = []
for idx,df in df_all.groupby(by="lamina_label"):
    df_tmp = df.iloc[[0,-1],:].reset_index(drop=True)
    df_tmp["Duration(hour)"] = [0,2]
    df_list.append(df_tmp)

In [ ]:
df_se = pd.concat(df_list).reset_index(drop=True)
df_se.iloc[:5,-3:]

In [ ]:
axis_labels = ["basal_vec","pca1","pca2","pca3","bv_w"]

for l in axis_labels:
    df_se[f"{l}_azimuth(deg)"] = np.degrees(np.arctan2(df_se[f"{l}_x2"],df_se[f"{l}_y2"]))
    df_se[f"{l}_elevation(deg)"] = df_se[f"{l}_z2"].apply(np.arcsin).apply(np.degrees)

col_labels = ["pca3_elevation(deg)","bv_w_elevation(deg)","basal_vec_elevation(deg)"]

title_labels = ["Lamina AdAb-Axis Elev (deg)",
                "Lamina Base AdAb-Axis Elev (deg)",
                "Lamina Base PD-Axis Elev (deg)"]

fig,axes = plt.subplots(
    figsize=(15,5),ncols=3,nrows=1
)
axes = axes.ravel()

plt.subplots_adjust(wspace=0.3,hspace=0.3)

for i in range(len(col_labels)):
    df_tmp = pd.pivot_table(df_se,values=col_labels[i],
                            columns="Duration(hour)",index="lamina_label").reset_index()
    df_tmp["Diff Angle (deg)"] = df_tmp[2] - df_tmp[0]
    kmeans = KMeans(n_clusters=2, random_state=0, n_init="auto")
    kmeans.fit(df_tmp[[0,2]].values)

    df_tmp["cluster no"] = kmeans.labels_

    ax = axes[i]
    sns.scatterplot(
        data = df_tmp,
        x=0,
        y=2,
        hue="cluster no",
        style="cluster no",
        s=100,
        legend=False,
        ax=ax
    )

    ax.plot([-75,50],[-75,50],"k--")
    ax.set_title(title_labels[i])
    ax.set_xlabel("0 hour")
    ax.set_ylabel("2 hour")
    ax.set_aspect("equal")
plt.show()

In [ ]:
df_se["Successful(1)/Incomplete(0)"] = df_se["lamina_label"].isin(succeed_idx)

In [ ]:
df_se[(df_se["Successful(1)/Incomplete(0)"]==1)&(df_se['Duration(hour)']==0)]["pca3_elevation(deg)"]

In [ ]:
df_se[(df_se["Successful(1)/Incomplete(0)"]==0)&(df_se['Duration(hour)']==0)]["pca3_elevation(deg)"]

In [ ]:
df_se[(df_se["Successful(1)/Incomplete(0)"]==1)&(df_se['Duration(hour)']==2)]["pca3_elevation(deg)"]

In [ ]:
axis_labels = ["basal_vec","pca1","pca2","pca3","bv_w"]

for l in axis_labels:
    df_se[f"{l}_azimuth(deg)"] = np.degrees(np.arctan2(df_se[f"{l}_x2"],df_se[f"{l}_y2"]))
    df_se[f"{l}_elevation(deg)"] = df_se[f"{l}_z2"].apply(np.arcsin).apply(np.degrees)

col_labels = ["bv_w_elevation(deg)","basal_vec_elevation(deg)"]

title_labels = ["AdAb-Axis Elev (deg)","PD-Axis Elev (deg)"]

plt.rcParams["font.size"] = 18
fig,axes = plt.subplots(figsize=(21,4),ncols=4,nrows=1)
plt.subplots_adjust(wspace=0.4)
axes = axes.ravel()

plt.subplots_adjust(wspace=0.3,hspace=0.3)

for i in range(len(col_labels)):
    df_tmp = pd.pivot_table(df_se,values=col_labels[i],
                            columns="Duration(hour)",index="lamina_label").reset_index()
    df_tmp["Diff Angle (deg)"] = df_tmp[2] - df_tmp[0]
    kmeans = KMeans(n_clusters=2, random_state=0, n_init="auto")
    kmeans.fit(df_tmp[[0,2]].values)

    df_tmp["cluster no"] = kmeans.labels_
    df_tmp["Successful(1)/Incomplete(0)"] = df_tmp["lamina_label"].isin(succeed_idx)

    ax = axes[i*2]
    sns.scatterplot(
        data = df_tmp,
        x=0,
        y=2,
        hue="Successful(1)/Incomplete(0)",
        s=100,
        legend=False,
        ax=ax
    )

    ax.set_title(title_labels[i])
    ax.set_xlabel("0 hour")
    ax.set_ylabel("2 hour")
    ax.set_xticks(np.arange(-90,91,30))
    ax.set_yticks(np.arange(-90,91,30))
    ax.set_xlim(-75,75)
    ax.set_ylim(-75,75)
    ax.set_aspect("equal")

    ax = axes[i*2+1]
    sns.boxplot(
        data = df_se,
        x="Duration(hour)",
        y=col_labels[i],
        hue="Successful(1)/Incomplete(0)",
        legend=False,
        ax=ax
    )
    ax.set_ylabel(title_labels[i])
#fig.savefig("graph/20250922_grav_reori_elev_angles.svg",format="svg",
#            bbox_inches="tight")
plt.show()

In [ ]:
df_cluster = pd.DataFrame(df_tmp[["lamina_label","cluster no"]].values,
             columns=["lamina_label","cluster no"])
df_all2 = df_all.merge(df_cluster,on="lamina_label")
df_se2 = df_se.merge(df_cluster,on="lamina_label")

In [ ]:
label = df_all["lamina_label"].unique()[2]
df_0704_3 = df_all[df_all["lamina_label"]==label]
df_0704_3.head()

Calculate R(t) and A(t)

In [ ]:
onb_cols = ["basal_vec_x2","basal_vec_y2","basal_vec_z2", #PD axis
                "pca2_x2","pca2_y2","pca2_z2", #ML axis
                "bv_w_x2","bv_w_y2","bv_w_z2"] #AdAb axis

shortest_path_list = []
rotations_list = []

for i in range(len(df_all["lamina_label"].unique())):
    label = df_all["lamina_label"].unique()[i]
    df_tmp = df_all[df_all["lamina_label"]==label]

    onb_list = []
    for i in range(len(df_tmp)):
        onb = df_tmp.reset_index(drop=True).loc[i,onb_cols].values.reshape(3,3).T
        onb_list.append(onb)
    rotations = R.from_matrix(onb_list) 
    t_values = df_tmp["Duration(min)"].values
    t_interp = np.arange(0, t_values.max(), 1)

    z_rot_angles = np.radians(np.arange(-180,180,0.1))

    azi_angles = []
    path_lens = []
    z_goal_sim_list = []

    for i in range(len(z_rot_angles)):
        z_rot = R.from_rotvec(np.array([0,0,1.0])*z_rot_angles[i])
        z_goal_sim = z_rot * rotations[-1]
        z_goal_sim_pd = z_goal_sim.as_matrix().T[0]
        azi = np.degrees(np.arctan2(z_goal_sim_pd[1],z_goal_sim_pd[0]))
        path_len = (z_goal_sim.inv()*rotations[0]).magnitude()
        azi_angles.append(azi)
        path_lens.append(path_len)
        z_goal_sim_list.append(z_goal_sim)

    spline = RotationSpline(t_values,rotations)
    R_interp = spline(t_interp)
    start_pd = rotations[0].as_matrix().T[0]
    start_adab = rotations[0].as_matrix().T[2]
    start_azi = np.degrees(np.arctan2(start_pd[1],start_pd[0]))
    end_pd = rotations[-1].as_matrix().T[0]
    end_adab = rotations[-1].as_matrix().T[2]
    true_end_azi = np.degrees(np.arctan2(end_pd[1],end_pd[0]))
    shortest_path_azi = azi_angles[np.array(path_lens).argmin()]

    shortest_path_dict = {
        "lamina_label":label,
        "Initial_PD_elevation(deg)":np.degrees(np.arcsin(start_pd[2])),
        "Final_PD_elevation(deg)":np.degrees(np.arcsin(end_pd[2])),
        "Initial_AdAb_elevation(deg)":np.degrees(np.arcsin(start_adab[2])),
        "Final_AdAb_elevation(deg)":np.degrees(np.arcsin(end_adab[2])),
        "Initial_PD_azimuth(deg)":start_azi,
        "Final_PD_azimuth(deg)":true_end_azi,
        "Shortest-path_PD_azimuth(deg)":shortest_path_azi,
        "Shortest-path_len":np.array(path_lens).min(),
        "Final_path_len":(rotations[-1].inv()*rotations[0]).magnitude(),
        "Series_path_len(linear)":np.sum((rotations.inv()[1:]*rotations[:-1]).magnitude()),
        "Series_path_len(spline)":np.sum((R_interp.inv()[1:]*R_interp[:-1]).magnitude())
    }

    shortest_path_list.append(shortest_path_dict)

    shortest_rot = z_goal_sim_list[np.array(path_lens).argmin()]
    t_se = [t_values[0],t_values[-1]]
    slerp0 = Slerp(t_se,R.concatenate([rotations[0],shortest_rot])) #closest in manuscript
    slerp1 = Slerp(t_se,R.concatenate([rotations[0],rotations[-1]])) #shortest in manuscript
    slerp2 = Slerp(t_values,rotations) #interpSLERP in manuscript

    rotations_dict = {
        "lamina_label":label,
        "rotations":rotations,
        "t_values":t_values,
        "t_interp":t_interp,
        "shortest_path":slerp0(t_interp),
        "final_path":slerp1(t_interp),
        "series_path_slerp":slerp2(t_interp),
        "series_path_spline":spline(t_interp),
        "delta_shortest_path":slerp0(t_interp)[:-1].inv()*slerp0(t_interp)[1:], #closest: A(t) = R(t)^{-1}*R(t+1) in manuscript
        "delta_final_path":slerp1(t_interp)[:-1].inv()*slerp1(t_interp)[1:], #shortest: A(t) = R(t)^{-1}*R(t+1) in manuscript
        "delta_series_path_slerp":slerp2(t_interp)[:-1].inv()*slerp2(t_interp)[1:], #interpSLERP: A(t) = R(t)^{-1}*R(t+1) in manuscript
        "delta_series_path_spline":spline(t_interp)[:-1].inv()*spline(t_interp)[1:]
    }
    rotations_list.append(rotations_dict)

In [ ]:
shortest_path_df = pd.DataFrame(shortest_path_list)
shortest_path_df.head()

In [ ]:
shortest_path_df["Successful(1)/Incomplete(0)"] = shortest_path_df["Final_AdAb_elevation(deg)"]>=0
shortest_path_df_selected = shortest_path_df[["lamina_label","Shortest-path_len","Final_path_len","Series_path_len(linear)","Series_path_len(spline)",
                                              "Initial_AdAb_elevation(deg)","Initial_PD_elevation(deg)","Initial_PD_azimuth(deg)","Final_PD_azimuth(deg)",
                                              "Successful(1)/Incomplete(0)"]]
shortest_path_df_selected.columns = ["lamina_label","closest","shortest","interpSLERP","interpSpline",
                                     "Initial_AdAb_elevation(deg)","Initial_PD_elevation(deg)","Initial_PD_azimuth(deg)","Final_PD_azimuth(deg)",
                                     "Successful(1)/Incomplete(0)"]
shortest_path_df_selected["shortest/closest"] = shortest_path_df_selected["shortest"]/shortest_path_df_selected["closest"] 
shortest_path_df_selected["interpSLERP/shortest"] = shortest_path_df_selected["interpSLERP"]/shortest_path_df_selected["shortest"] 
shortest_path_df_selected.to_csv("20250922_path_len_summary.csv")
shortest_path_df_selected.groupby(by="Successful(1)/Incomplete(0)").describe().T
shortest_path_df["Successful(1)/Incomplete(0)"] = shortest_path_df["Final_AdAb_elevation(deg)"]>=0
shortest_path_df_selected = shortest_path_df[["lamina_label","Shortest-path_len","Final_path_len","Series_path_len(linear)","Series_path_len(spline)",
                                              "Initial_AdAb_elevation(deg)","Initial_PD_elevation(deg)","Initial_PD_azimuth(deg)","Final_PD_azimuth(deg)",
                                              "Successful(1)/Incomplete(0)"]]
shortest_path_df_selected.columns = ["lamina_label","closest","shortest","interpSLERP","interpSpline",
                                     "Initial_AdAb_elevation(deg)","Initial_PD_elevation(deg)","Initial_PD_azimuth(deg)","Final_PD_azimuth(deg)",
                                     "Successful(1)/Incomplete(0)"]
shortest_path_df_selected["shortest/closest"] = shortest_path_df_selected["shortest"]/shortest_path_df_selected["closest"] 
shortest_path_df_selected["interpSLERP/shortest"] = shortest_path_df_selected["interpSLERP"]/shortest_path_df_selected["shortest"] 
shortest_path_df_selected.to_csv("20250922_path_len_summary.csv")
shortest_path_df_selected.groupby(by="Successful(1)/Incomplete(0)").describe().T

In [ ]:
shortest_path_df_selected

In [ ]:
shortest_path_df["Succeeded/Failed"] = shortest_path_df["Final_AdAb_elevation(deg)"]>=0
shortest_path_df["Shortest/Final"] = shortest_path_df["Shortest-path_len"]/shortest_path_df["Final_path_len"]
shortest_path_df["Series/Final"] = shortest_path_df["Series_path_len(linear)"]/shortest_path_df["Final_path_len"]
shortest_path_df["abs_Initial_PD_azimuth(deg)"] = shortest_path_df["Initial_PD_azimuth(deg)"].apply(np.abs)
shortest_path_df["abs_Final_PD_azimuth(deg)"] = shortest_path_df["Final_PD_azimuth(deg)"].apply(np.abs)

sns.pairplot(
    data=shortest_path_df,
    hue="Succeeded/Failed"
)
#plt.savefig("graph/20250916_pairplot.png",dpi=300,bbox_inches="tight")
plt.show()

Draw 3D plots

In [ ]:
elev,azim = 90,0
i = 4
label = rotations_list[i]["lamina_label"]

for col in ["final_path","series_path_slerp","series_path_spline","shortest_path"]:
        interp_mtx = rotations_list[i][col].as_matrix()
        px,py,pz = df_all[df_all["lamina_label"]==rotations_list[i]["lamina_label"]][["p_x2","p_y2","p_z2"]].mean(axis=0)

        savedir = f"graph/20250916/{label}/{col}"
        print(savedir)
        os.makedirs(savedir,exist_ok=True)
        for n in range(len(interp_mtx)):

                fig = plt.figure(figsize=(10, 10))
                ax = fig.add_subplot(111, projection='3d')

                u, v = np.meshgrid(np.linspace(0, 2*np.pi, 50),
                                np.linspace(0, np.pi, 50))
                sphere_x = np.cos(u) * np.sin(v)
                sphere_y = np.sin(u) * np.sin(v)
                sphere_z = np.cos(v)
                ax.plot_surface(sphere_x, sphere_y, sphere_z, color='lightblue', alpha=0.1, edgecolor='none')

                v0 = np.array([0, 0, 0])
                v1 = np.array([interp_mtx[n, 0, 0], interp_mtx[n, 1, 0], interp_mtx[n, 2, 0]])
                v2 = np.array([(interp_mtx[n, 0, 0]+interp_mtx[n, 0, 1])/2,
                        (interp_mtx[n, 1, 0]+interp_mtx[n, 1, 1])/2,
                        (interp_mtx[n, 2, 0]+interp_mtx[n, 2, 1])/2])
                v3 = np.array([(interp_mtx[n, 0, 0]-interp_mtx[n, 0, 1])/2,
                        (interp_mtx[n, 1, 0]-interp_mtx[n, 1, 1])/2,
                        (interp_mtx[n, 2, 0]-interp_mtx[n, 2, 1])/2])

                triangle = [v0, v3, v1, v2]
                triangle_poly = Poly3DCollection([triangle], alpha=0.5, color='darkgreen')
                ax.add_collection3d(triangle_poly)

                ax.plot(
                        [interp_mtx[n, 0, 0]/2,(interp_mtx[n, 0, 0]+interp_mtx[n, 0, 1])/2],
                        [interp_mtx[n, 1, 0]/2,(interp_mtx[n, 1, 0]+interp_mtx[n, 1, 1])/2],
                        [interp_mtx[n, 2, 0]/2,(interp_mtx[n, 2, 0]+interp_mtx[n, 2, 1])/2],
                        color='darkgreen',
                        linewidth=2
                        )

                for i2 in range(3):
                        ax.plot(
                                [0,interp_mtx[n, 0, i2]],
                                [0,interp_mtx[n, 1, i2]],
                                [0,interp_mtx[n, 2, i2]],
                                color='black',
                                linestyle='dotted',
                                linewidth=2
                                )
                ax.plot(
                        [-px,0],
                        [-py,0],
                        [-pz,0],
                        color='darkolivegreen',
                        linewidth=4,
                        alpha=0.6
                )

                for j in range(3):
                        x_interp = interp_mtx[:, 0, j]
                        y_interp = interp_mtx[:, 1, j]
                        z_interp = interp_mtx[:, 2, j]

                        ax.plot(x_interp, y_interp, z_interp, linewidth=2)
                ax.set_xlabel("X")
                ax.set_ylabel("Y")
                ax.set_zlabel("Z")
                ax.set_title(f"{label}: Duration Time: {n:04d} min")

                ax.set_xlim([-1.1, 1.1])
                ax.set_ylim([-1.1, 1.1])
                ax.set_zlim([-1.1, 1.1])
                ax.set_box_aspect([1, 1, 1]) 

                ax.view_init(elev=elev, azim=azim)

                #ax.legend()
                #fig.savefig(f"{savedir}/{n:05d}.png",dpi=150,bbox_inches="tight")
                plt.close()

                #img = cv2.imread(f"{savedir}/{n:05d}.png")
                #cv2.imwrite(f"{savedir}/{n:05d}.png",img[:img.shape[0]//2*2,:img.shape[1]//2*2,:])

In [ ]:
succeed_idx = 0,1,2,4,8
fail_idx = 3,5,6,7,9

Swing-Twist Decomposition

In [ ]:
angvel_xyz_list = []
keys_list = ['shortest_path', 'final_path', 'series_path_slerp','series_path_spline']
label_list = ['Closest', 'Shortest', 'Interp SLERP','Interp Spline']

for idx in range(10):
    for i,key in enumerate(keys_list):
        tmpR = rotations_list[idx][key]
        delta_tmpR = tmpR[:-1].inv()*tmpR[1:]
        x0,y0,z0 = delta_tmpR.as_rotvec().T
        yz = np.sqrt(y0**2+z0**2).sum()
        x,y,z = np.abs(delta_tmpR.as_rotvec()).sum(axis=0)
        x2,y2,z2 = delta_tmpR.as_rotvec().sum(axis=0)
        path_len = np.sum(delta_tmpR.magnitude())

        angvel_xyz_dict = {
            "idx":idx,
            "lamina_label":rotations_list[idx]["lamina_label"],
            "path":label_list[i],
            "Twist(X)":x,
            "Swing(Y,E)":y,
            "Swing(Z,A)":z,
            "Swing(YZ)":yz,
            "Twist(X)_signed":x2,
            "Swing(Y,E)_signed":y2,
            "Swing(Z,A)_signed":z2,
            "PathLength":path_len}
        angvel_xyz_list.append(angvel_xyz_dict)


In [ ]:
angvel_xyz_df = pd.DataFrame(angvel_xyz_list)
angvel_xyz_df.head()

In [ ]:
angvel_xyz_df = angvel_xyz_df.merge(shortest_path_df[["lamina_label","Successful(1)/Incomplete(0)"]],on="lamina_label")

In [ ]:
for col in angvel_xyz_df.columns[3:-1]:
    angvel_xyz_df[col+" (deg)"] = angvel_xyz_df[col].apply(np.degrees)
angvel_xyz_df["Log2(Swing/Twist)"] = np.log2(angvel_xyz_df["Swing(YZ)"]/angvel_xyz_df["Twist(X)"])

In [ ]:
sns.set_style("whitegrid")
fig,axes = plt.subplots(
    figsize=(21,4),ncols=4,nrows=1
)
plt.subplots_adjust(wspace=0.3)

for i in range(3):
    ax = axes[i]
    key = label_list[i]
    sns.scatterplot(
        data=angvel_xyz_df[angvel_xyz_df["path"]==key],
        x="Twist(X)",
        y="Swing(YZ)",
        c="gray",
        style="Successful(1)/Incomplete(0)",
        s=80,
        alpha=0.8,
        legend=None,
        ax=ax
    )
    ax.set_ylim(-0.05,np.pi*2/3)
    ax.set_xlim(-0.05,np.pi*1/3)
    ax.set_title(key)

ax = axes[3]
sns.scatterplot(
    data=angvel_xyz_df[angvel_xyz_df["path"].isin(label_list[:3])],
    x="Twist(X)",
    y="Swing(YZ)",
    hue="path",
    style="Successful(1)/Incomplete(0)",
    s=80,
    alpha=0.8,
    ax=ax
)
ax.set_ylim(-0.05,np.pi*2/3)
ax.set_xlim(-0.05,np.pi*1/3)
ax.set_title("Merged")
ax.legend(loc="upper left",bbox_to_anchor=(1.02,1))
#fig.savefig("graph/20250918_path_swing_twist_ratio.svg",format="svg",bbox_inches="tight")
plt.show()

In [ ]:
angvel_xyz_df.groupby(by="path").max(numeric_only=True)

In [ ]:
angvel_xyz_df.groupby(by="path").min(numeric_only=True)

In [ ]:
df_tmp = angvel_xyz_df[angvel_xyz_df["path"].isin(label_list[1:3])]
df_tmp = df_tmp[df_tmp["Successful(1)/Incomplete(0)"]==1]
df_tmp = df_tmp[["lamina_label","path","Twist(X)","Swing(YZ)","PathLength"]]
df_shortest = df_tmp[df_tmp["path"]=="Shortest"]
df_slerp = df_tmp[df_tmp["path"]=="Interp SLERP"]
detour_ratio = df_slerp["PathLength"].values/df_shortest["PathLength"].values-1
swing_ratio = df_shortest["Swing(YZ)"].values/(df_shortest["Swing(YZ)"].values+df_shortest["Twist(X)"].values)
print(detour_ratio)
print(swing_ratio)

Calculate detour ratio and swing ratio

In [ ]:
df_tmp = angvel_xyz_df[angvel_xyz_df["path"].isin(label_list[1:3])]
df_tmp = df_tmp[df_tmp["Successful(1)/Incomplete(0)"]==1]
df_tmp = df_tmp[["lamina_label","path","Twist(X)","Swing(YZ)","PathLength"]]
df_shortest = df_tmp[df_tmp["path"]=="Shortest"]
df_slerp = df_tmp[df_tmp["path"]=="Interp SLERP"]
detour_ratio = df_slerp["PathLength"].values/df_shortest["PathLength"].values-1
swing_ratio = df_shortest["Swing(YZ)"].values/(df_shortest["Swing(YZ)"].values+df_shortest["Twist(X)"].values)
print(detour_ratio)
print(swing_ratio)

In [ ]:
from scipy import stats
stats.pearsonr(swing_ratio,detour_ratio)

In [ ]:
results = stats.linregress(swing_ratio,detour_ratio)
print(results)

In [ ]:
plt.scatter(swing_ratio,detour_ratio,c="blue")
x = np.array([0.5,1.0])
plt.plot(x,x*results.slope+results.intercept,"k--",alpha=0.5)
plt.title(rf"$y=${results.slope:.02f}$x+${results.intercept:.02f}  $r=${results.rvalue:.03f}  $p=${results.pvalue:.03f}")
plt.xlabel("Swing ratio in Shortest path")
plt.ylabel("Detour ratio")
#plt.savefig("graph/20250923_path_swing_detour.svg",format="svg",bbox_inches="tight")
plt.show()